# Performance Testing — Edu Game AI Service

Benchmark the LangGraph pipeline and API for latency & throughput.

**Goals:**
- Single generation latency (target: < 60s for 5 questions)
- Batch latency across 5 STEM topics
- Concurrent request handling (5 simultaneous)
- Per-node timing breakdown (supervisor → math_agent → reviewer → formatter)

**Modes:**
- **Direct pipeline** — invoke LangGraph directly (no HTTP)
- **HTTP API** — test via local Docker container at `localhost:8000`

## 1. Setup

In [ ]:
import sys, time, json, asyncio, statistics
from pathlib import Path

project_root = str(Path.cwd().parent.parent)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f'Project root: {project_root}')

In [ ]:
from src.config import get_settings
from src.api.schemas import GenerationRequest, GameType, DifficultyLevel, DocScope
from src.graph.builder import get_graph_app

settings = get_settings()
print(f'Environment: {settings.env}')
print(f'Generation model: {settings.generation_model}')
print(f'Review model: {settings.review_model}')

## 2. Test Configuration

In [ ]:
TEST_TOPICS = [
    'Đạo hàm',
    'Phương trình bậc hai',
    'Định luật Newton',
    'Phản ứng oxi hóa khử',
    'Tích phân',
]

NUM_QUESTIONS = 5
DIFFICULTY = DifficultyLevel.MEDIUM
TARGET_LATENCY_S = 60  # seconds per generation
CONCURRENT_REQUESTS = 5

# Docker API base URL (for HTTP tests)
API_BASE = 'http://localhost:8000'

print(f'Topics: {len(TEST_TOPICS)}')
print(f'Questions per topic: {NUM_QUESTIONS}')
print(f'Target latency: {TARGET_LATENCY_S}s')
print(f'Concurrent requests: {CONCURRENT_REQUESTS}')

## 3. Direct Pipeline — Single Topic Benchmark

Invoke the LangGraph pipeline directly (no HTTP overhead) to measure pure generation time.

In [ ]:
topic = TEST_TOPICS[0]
request = GenerationRequest(
    user_id='perf_test',
    topic=topic,
    game_types=[GameType.QUIZ],
    num_questions=NUM_QUESTIONS,
    difficulty=DIFFICULTY,
    doc_scope=DocScope.SYSTEM,
)

app = get_graph_app()
initial_state = {'request': request, 'doc_scope': request.doc_scope.value}

print(f'Running pipeline for "{topic}" ({NUM_QUESTIONS} questions)...')
t0 = time.perf_counter()
result = await app.ainvoke(initial_state)
elapsed = time.perf_counter() - t0

items = result.get('content_items', [])
final = result.get('final_output')

print(f'\n--- Single Topic Result ---')
print(f'Elapsed: {elapsed:.1f}s (target: <{TARGET_LATENCY_S}s) {"✓" if elapsed < TARGET_LATENCY_S else "✗ SLOW"}')
print(f'Content items generated: {len(items)}')
print(f'Errors: {result.get("errors", [])}')
if final:
    print(f'Metadata: {final.metadata.generation_time_seconds:.1f}s (formatter-reported)')

## 4. Direct Pipeline — Batch Benchmark (Sequential)

Run all 5 topics sequentially to measure total throughput.

In [ ]:
timings = []
app = get_graph_app()

for topic in TEST_TOPICS:
    request = GenerationRequest(
        user_id='perf_test',
        topic=topic,
        game_types=[GameType.QUIZ],
        num_questions=NUM_QUESTIONS,
        difficulty=DIFFICULTY,
        doc_scope=DocScope.SYSTEM,
    )
    t0 = time.perf_counter()
    result = await app.ainvoke({'request': request, 'doc_scope': request.doc_scope.value})
    elapsed = time.perf_counter() - t0
    items_count = len(result.get('content_items', []))
    errors = result.get('errors', [])
    status = '✓' if elapsed < TARGET_LATENCY_S and not errors else '✗'
    timings.append({'topic': topic, 'elapsed': elapsed, 'items': items_count, 'errors': errors})
    print(f'{status} {topic:<30} {elapsed:>6.1f}s  {items_count} items')

print(f'\n--- Batch Summary ---')
times = [t['elapsed'] for t in timings]
print(f'Total: {sum(times):.1f}s')
print(f'Mean:  {statistics.mean(times):.1f}s')
print(f'Stdev: {statistics.stdev(times):.1f}s' if len(times) > 1 else '')
print(f'Min:   {min(times):.1f}s')
print(f'Max:   {max(times):.1f}s')
passed = sum(1 for t in timings if t['elapsed'] < TARGET_LATENCY_S and not t['errors'])
print(f'Pass rate: {passed}/{len(timings)} within {TARGET_LATENCY_S}s target')

## 5. Direct Pipeline — Concurrent Benchmark

Run multiple topics concurrently to simulate real load.

In [ ]:
async def run_one_pipeline(topic: str) -> dict:
    request = GenerationRequest(
        user_id='perf_concurrent',
        topic=topic,
        game_types=[GameType.QUIZ],
        num_questions=NUM_QUESTIONS,
        difficulty=DIFFICULTY,
        doc_scope=DocScope.SYSTEM,
    )
    app = get_graph_app()
    t0 = time.perf_counter()
    try:
        result = await app.ainvoke({'request': request, 'doc_scope': request.doc_scope.value})
        elapsed = time.perf_counter() - t0
        return {'topic': topic, 'elapsed': elapsed, 'items': len(result.get('content_items', [])), 'ok': True}
    except Exception as e:
        elapsed = time.perf_counter() - t0
        return {'topic': topic, 'elapsed': elapsed, 'items': 0, 'ok': False, 'error': str(e)}

print(f'Running {CONCURRENT_REQUESTS} topics concurrently...')
t0 = time.perf_counter()
concurrent_results = await asyncio.gather(
    *[run_one_pipeline(t) for t in TEST_TOPICS[:CONCURRENT_REQUESTS]]
)
total_wall = time.perf_counter() - t0

print(f'\n--- Concurrent Results ---')
for r in concurrent_results:
    status = '✓' if r['ok'] else '✗'
    print(f'{status} {r["topic"]:<30} {r["elapsed"]:>6.1f}s  {r["items"]} items')
print(f'\nWall-clock time: {total_wall:.1f}s (vs {sum(t["elapsed"] for t in concurrent_results):.1f}s sequential)')

---

## 6. HTTP API — Docker Container Tests

**Prerequisites:** Docker container running at `localhost:8000`

```bash
docker compose up -d
```

In [ ]:
import httpx

# Check if Docker container is running
try:
    resp = httpx.get(f'{API_BASE}/health', timeout=5)
    print(f'✓ Docker API reachable: {resp.json()}')
    DOCKER_AVAILABLE = True
except Exception as e:
    print(f'✗ Docker API not reachable at {API_BASE}: {e}')
    print('  Start with: docker compose up -d')
    DOCKER_AVAILABLE = False

In [ ]:
if not DOCKER_AVAILABLE:
    print('⚠️  Skipping HTTP tests — Docker not running')
else:
    payload = {
        'user_id': 'perf_http_test',
        'topic': 'Đạo hàm',
        'game_types': ['quiz'],
        'num_questions': 5,
        'difficulty': 'medium',
        'doc_scope': 'system',
    }

    t0 = time.perf_counter()
    with httpx.Client(base_url=API_BASE, timeout=120) as client:
        # POST /generate
        resp = client.post('/api/v1/generate', json=payload)
        assert resp.status_code == 202, f'Expected 202, got {resp.status_code}: {resp.text}'
        job = resp.json()
        request_id = job['request_id']
        print(f'Job created: {request_id}')

        # Poll until completed/failed
        for attempt in range(60):
            time.sleep(2)
            poll = client.get(f'/api/v1/generations/{request_id}')
            status = poll.json().get('status')
            if status in ('completed', 'failed'):
                break
        else:
            status = 'timeout'

    elapsed = time.perf_counter() - t0
    print(f'\n--- HTTP API Benchmark ---')
    print(f'Status: {status}')
    print(f'Elapsed: {elapsed:.1f}s (target: <{TARGET_LATENCY_S}s)')
    if status == 'completed':
        content = poll.json().get('content', {})
        meta = content.get('metadata', {})
        print(f'Pipeline time: {meta.get("generation_time_seconds", 0):.1f}s')
        print(f'Quiz items: {len(content.get("content", {}).get("quiz", []))}')

## 7. HTTP API — Concurrent Requests

In [ ]:
if not DOCKER_AVAILABLE:
    print('⚠️  Skipping — Docker not running')
else:
    async def http_generate(client: httpx.AsyncClient, topic: str) -> dict:
        payload = {
            'user_id': 'perf_concurrent_http',
            'topic': topic,
            'game_types': ['quiz'],
            'num_questions': 5,
            'difficulty': 'medium',
            'doc_scope': 'system',
        }
        t0 = time.perf_counter()
        resp = await client.post('/api/v1/generate', json=payload)
        request_id = resp.json()['request_id']

        for _ in range(60):
            await asyncio.sleep(2)
            poll = await client.get(f'/api/v1/generations/{request_id}')
            status = poll.json().get('status')
            if status in ('completed', 'failed'):
                break
        else:
            status = 'timeout'

        elapsed = time.perf_counter() - t0
        return {'topic': topic, 'status': status, 'elapsed': elapsed}

    print(f'Sending {CONCURRENT_REQUESTS} concurrent HTTP requests...')
    t0 = time.perf_counter()
    async with httpx.AsyncClient(base_url=API_BASE, timeout=180) as client:
        results = await asyncio.gather(
            *[http_generate(client, t) for t in TEST_TOPICS[:CONCURRENT_REQUESTS]]
        )
    wall = time.perf_counter() - t0

    print(f'\n--- Concurrent HTTP Results ---')
    for r in results:
        s = '✓' if r['status'] == 'completed' else '✗'
        print(f'{s} {r["topic"]:<30} {r["elapsed"]:>6.1f}s  [{r["status"]}]')
    print(f'\nWall-clock: {wall:.1f}s')

## 8. Summary

In [ ]:
print('=' * 60)
print('PERFORMANCE TEST SUMMARY')
print('=' * 60)
print(f'Model: {settings.generation_model}')
print(f'Questions per topic: {NUM_QUESTIONS}')
print(f'Target latency: <{TARGET_LATENCY_S}s')
print()

if timings:
    times = [t['elapsed'] for t in timings]
    passed = sum(1 for t in timings if t['elapsed'] < TARGET_LATENCY_S)
    print(f'[Direct Pipeline — Sequential]')
    print(f'  Mean:   {statistics.mean(times):.1f}s')
    print(f'  Median: {statistics.median(times):.1f}s')
    print(f'  P95:    {sorted(times)[int(len(times)*0.95)]:.1f}s')
    print(f'  Pass:   {passed}/{len(timings)}')
    print()

if 'concurrent_results' in dir():
    ctimes = [r['elapsed'] for r in concurrent_results]
    print(f'[Direct Pipeline — Concurrent ({CONCURRENT_REQUESTS}x)]')
    print(f'  Wall-clock: {total_wall:.1f}s')
    print(f'  Mean per-task: {statistics.mean(ctimes):.1f}s')
    print(f'  OK: {sum(1 for r in concurrent_results if r["ok"])}/{len(concurrent_results)}')

print(f'\nDocker API: {"Available" if DOCKER_AVAILABLE else "Not tested"}')